# Notebook: 03 Training Test
### Purpose: create train and validation datasets, build augmentations, run Trainer, save versioned checkpoints.


In [1]:
import os
import sys


sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

import random

import torch

from src.data.annotations import load_json_annotations
from src.data.augmentations import get_train_augmentations, get_val_augmentations
from src.data.loaders import ImageMaskDataset
from src.models.zoo import MODEL_BUILDERS
from src.training.engine import run_training
from src.utils.config import Config
from src.utils.helpers import init_notebook, p


config = Config.load()

init_notebook(config.train.seed)

train_dir = config.paths.train_images
annotations_path = config.paths.annotations
entries = load_json_annotations(annotations_path)

# Shuffle entries
random.shuffle(entries)



=== init_notebook ===
Done


#### Dataset split

In [2]:
# Compute number of validation samples (20 percent of dataset)
val_count = max(1, int(0.2 * len(entries)))

# Split validation set, and training set
val_entries = entries[:val_count]
train_entries = entries[val_count:]

# Build augmentation pipelines for training and validation
train_tf = get_train_augmentations(config.train.image_size)
val_tf = get_val_augmentations(config.train.image_size)

# Build dataset objects that load image-mask pairs and apply transforms
train_ds = ImageMaskDataset(train_entries, train_dir, transform = train_tf)
val_ds = ImageMaskDataset(val_entries, train_dir, transform = val_tf)

# Build dataloaders
train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size = config.train.batch_size,
        shuffle = True,
        num_workers = config.train.num_workers,
)

val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size = config.train.batch_size,
        shuffle = False,
        num_workers = config.train.num_workers,
)

p("Train samples", len(train_ds))
p("Val samples", len(val_ds))

Train samples: 120
Val samples: 30


#### Available Models

In [3]:
p("Models", MODEL_BUILDERS)
#config.show()
p("Batch", config.train.batch_size)
p("Epochs", config.train.epochs)
p("Learning Rate", config.train.learning_rate, precision = 9)
p("Image Size", config.train.image_size)


Models: 8 keys
  simple_cnn: <function create_simple_cnn at 0x000002C595D25E40>
  unet: <function create_unet at 0x000002C595D24040>
  smp_unet: <function create_smp_unet at 0x000002C595D25DA0>
  smp_fpn: <function create_smp_fpn at 0x000002C58EA84040>
  smp_linknet: <function create_smp_linknet at 0x000002C58EA776A0>
  smp_deeplabv3: <function create_smp_deeplabv3 at 0x000002C595C61800>
  smp_deeplabv3plus: <function create_smp_deeplabv3plus at 0x000002C595C602C0>
  segformer: <function create_segformer at 0x000002C5A14C3600>
Batch: 1
Epochs: 2
Learning Rate: 0.010000000
Image Size: 32


#### Run Training

In [4]:
version_root = config.paths.models
trainer = run_training(
        config = config,
        train_loader = train_loader,
        val_loader = val_loader,
        version_root = version_root,
        model_name = "simple_cnn",
)

=== Training started ===
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
SimpleCNN                                [1, 3, 32, 32]            [1, 1, 32, 32]            --
├─Sequential: 1-1                        [1, 3, 32, 32]            [1, 32, 32, 32]           --
│    └─Sequential: 2-1                   [1, 3, 32, 32]            [1, 32, 32, 32]           --
│    │    └─Conv2d: 3-1                  [1, 3, 32, 32]            [1, 32, 32, 32]           896
│    │    └─ReLU: 3-2                    [1, 32, 32, 32]           [1, 32, 32, 32]           --
│    │    └─BatchNorm2d: 3-3             [1, 32, 32, 32]           [1, 32, 32, 32]           64
│    └─Sequential: 2-2                   [1, 32, 32, 32]           [1, 64, 32, 32]           --
│    │    └─Conv2d: 3-4                  [1, 32, 32, 32]           [1, 64, 32, 32]           18,496
│    │    └─ReLU: 3-5                    [1, 64, 32, 32]           [1, 64, 32, 32]           --
│    